# Moirai zero-shot + DTW neighbours

Upstream source: `Morai_working_zero_shot.ipynb` in the thesis workbench.

**Input**: `DTW_DFS_DIR / 'dtw_20_dfs' / *.parquet`.
**Output**: `moirai_results_dir('dtw20')` - `moirai_smoketest.csv`, per-experiment metrics CSVs and forecast CSVs (batched + FP16 ultra-fast sweep, past-only features, multivariate target_dim=20, and H=6 auto-scale runs).

**Inference only**: this notebook runs the pretrained Moirai-1.1-R-small checkpoint in zero-shot mode; no fine-tuning.


## Canonical cell for the paper

The paper reports the **zero-shot Moirai baseline** at the headline horizon (h = 6) from the cell titled *Moirai H=6 (auto-scales for 110-127 weeks)*. Earlier experiment cells (smoketest, past-only-features sweep, multivariate target_dim=20, rebuilt-with-improvements) are kept for transparency into the iterative process behind the reported number; they do not change it.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_20, moirai_results_dir,
)
from api_keys import get_hf_token  # noqa: E402

ensure_env()
_hf_token = get_hf_token()
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token


In [ ]:
# --- Pin a stable stack for Py3.11 + uni2ts forecast path ---
!pip -q uninstall -y numpy pandas scipy transformers tokenizers huggingface-hub lightning torchmetrics pillow

# Core ABI pair (NumPy 1.26 + SciPy 1.11) + pandas
!pip -q install --no-deps numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 pillow==11.0.0

# HF/Lightning stack that previously clashed
!pip -q install --no-deps transformers==4.46.2 tokenizers==0.22.0 huggingface-hub==0.36.0 lightning==2.3.3 torchmetrics==1.3.2

# (optional) make sure these are present
!pip -q install uni2ts==2.0.0 gluonts==0.14.3 polars==1.32.1 tqdm==4.66.5 pyarrow==22.0.0

# ---- Force runtime restart so NumPy/Pandas C-extensions reload cleanly ----
import os, signal, sys
os.kill(os.getpid(), 9)


In [ ]:
# Fix the HF mismatch: keep your current transformers, downgrade tokenizers
!pip -q uninstall -y tokenizers
!pip -q install --no-deps tokenizers==0.20.3

# sanity check
import tokenizers, transformers
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)


In [ ]:
# Drive is mounted by the bootstrap cell; no remount needed.

from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---- Paths (deduced from your snippet) ----
BASE   = DATA_ROOT
IN_DIR = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")

print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_smoketest.csv"

# ---- Device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
print("✅ Using device:", device)
print("Input dir:", IN_DIR)

# ---- Helpers (univariate-only; fast)
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path):
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path):
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        return None
    base = (
        df_pl
        .select(pl.col("week"),
                pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
        .filter(pl.col("y").is_not_null())
    )
    pdf = base.to_pandas()
    pdf["y"] = pdf["y"].astype(np.float32, copy=False)
    return pdf.sort_values("ds").reset_index(drop=True)

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5:
        return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape_canonical(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse_vec(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def build_listdataset_for_eval(train_df: pd.DataFrame, test_df: pd.DataFrame, freq: str="W"):
    start = pd.Timestamp(train_df["ds"].iloc[0])
    entry = {"start": start, "target": train_df["y"].to_numpy(dtype=np.float32)}
    return ListDataset([entry], freq=freq)

# ---- Model
MODEL_ID = "Salesforce/moirai-1.1-R-small"
module = MoiraiModule.from_pretrained(MODEL_ID).to(device).eval()

def run_one_univariate(pdf: pd.DataFrame, horizon: int, freq: str = "W"):
    tr, te = split_train_test(pdf, horizon)
    if tr is None:
        return np.nan, np.nan
    forecaster = MoiraiForecast(
        prediction_length=horizon,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
        context_length=int(np.clip(len(tr)//2, 32, len(tr))),
        module=module,
        patch_size="auto",
        num_samples=10,  # ultra-fast
    )
    predictor = forecaster.create_predictor(batch_size=16, device=device)
    ds = build_listdataset_for_eval(tr, te, "W")
    with torch.inference_mode():
        fc = next(predictor.predict(ds))
    yhat = np.asarray(fc.mean, dtype=np.float32)
    y    = te["y"].to_numpy(dtype=np.float32)
    if len(y) != horizon:
        y = y[-horizon:]
    return smape_canonical(y, yhat), rmse_vec(y, yhat)

# ---- Run on ≤2 files
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])[:2]
if not files:
    raise FileNotFoundError(f"No parquet/csv files found in {IN_DIR}")

rows = []
for p in files:
    try:
        pdf = prep_keyword_pdf(p)
        if pdf is None or len(pdf) < 16:
            print(f"Skipping {p.stem}: not enough data/cols")
            continue
        s, r = run_one_univariate(pdf, horizon=6)
        rows.append({"keyword": p.stem, "exog_setting": "univariate", "horizon_weeks": 6,
                     "smape_canonical": s, "rmse": r, "n_regressors": 0})
        print(f"✓ {p.stem}: sMAPE={s:.2f} | RMSE={r:.4f}")
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")

import pandas as pd
if rows:
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    print("\n✅ Smoke test complete.")
    print("Saved:", OUT_CSV)
else:
    print("\nNo results produced — check data and paths.")


In [ ]:
# Compute a robust context length per horizon from the preloaded pdfs
import numpy as np

# assumes pdfs (list of (name, pdf)) and HORIZONS already exist in the notebook
ctx_by_H = {}
for H in HORIZONS:
    train_lens = [len(pdf) - H for _, pdf in pdfs if len(pdf) > H + 5]
    if not train_lens:
        ctx_by_H[H] = 32
        continue
    # median(train_len)//2, clamped to [32, 512] and not exceeding 90th percentile of train length
    med = int(np.median(train_lens))
    p90 = int(np.percentile(train_lens, 90))
    ctx = med // 2
    ctx = int(np.clip(ctx, 32, min(512, p90)))
    ctx_by_H[H] = ctx

print("Context per horizon:", ctx_by_H)


In [ ]:
# ===== Ultra-fast Moirai sweep (batched + FP16) for T4 =====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch, gc
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ TUNABLES ------------
BEST_N     = 20
HORIZONS   = [1, 6, 12]
FREQ       = "W"
NUM_SAMPLES= 1        # fastest
BATCH_SIZE = 384     # try 384, drop to 192 if OOM
MAX_FILES  = None     # None = all
CHUNK_FLUSH= 5000     # CSV flush stride
USE_FP16   = True     # mixed-precision on T4
# ----------------------------------

# ------------ DEVICE --------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ PATHS ---------------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / f"moirai_multivariate_metrics_dtw{BEST_N}_full.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ HELPERS -------------
SMALL_COVARS = ['impressions_sum','n_dev_desktop','n_dev_mobile','n_dev_tablet',
                'n_st_branded_search','n_st_generic_search']
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str):
    w, y = wwyyyy.split("-")
    from datetime import date
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        return None
    base = (
        df_pl.select(pl.col("week"),
                     pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
             .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
             .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    )
    exog_cols = [c for c in df_pl.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]
    if exog_cols:
        exog = (df_pl.select(["week"] + exog_cols)
                    .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
                    .drop("week").sort("ds"))
        full = base.join(exog, on="ds", how="left")
        pdf = full.to_pandas()
        pdf[exog_cols] = pdf[exog_cols].ffill().bfill().astype(np.float32, errors="ignore")
        for c in exog_cols:
            if pdf[c].isna().any():
                med = pdf[c].median()
                pdf[c] = pdf[c].fillna(0.0 if pd.isna(med) else med)
    else:
        pdf = base.to_pandas()
    pdf["y"] = pdf["y"].astype(np.float32, copy=False)
    return pdf.sort_values("ds").reset_index(drop=True)

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def covars_univariate(pdf): return []
def covars_small(pdf):
    base = [c for c in SMALL_COVARS if c in pdf.columns]
    lastweek = [c for c in pdf.columns if c.startswith("cpc_lastweek__")]
    return base + lastweek
def covars_all(pdf): return [c for c in pdf.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]

EXOG_SETTINGS = [("univariate", covars_univariate),
                 ("small",      covars_small),
                 ("all",        covars_all)]

def smape_canonical(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse_vec(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

# ---------- LOAD DATA ONCE ----------
files_all = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
files = files_all if MAX_FILES is None else files_all[:MAX_FILES]
if not files: raise FileNotFoundError(f"No files found in {IN_DIR}")

pdfs = []
for p in tqdm(files, desc="Preloading"):
    pdf = prep_keyword_pdf(p)
    if pdf is not None and len(pdf) >= 16:
        pdfs.append((p.stem, pdf))
print(f"Loaded {len(pdfs)} series out of {len(files)}.")

# ---------- MODEL (FP16 ready) ----------
MODEL_ID = "Salesforce/moirai-1.1-R-small"
module = MoiraiModule.from_pretrained(MODEL_ID).to(device)
if USE_FP16 and device == "cuda":
    module = module.to(torch.float16)
module.eval()

# ===== Drop-in replacement: batched loop keyed by exact covariate signature =====
import gc
from collections import defaultdict

def covars_univariate(pdf): return []
def covars_small(pdf):
    base = [c for c in SMALL_COVARS if c in pdf.columns]
    lastweek = [c for c in pdf.columns if c.startswith("cpc_lastweek__")]
    return base + lastweek
def covars_all(pdf):
    return [c for c in pdf.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]

EXOG_SETTINGS = [
    ("univariate", covars_univariate),
    ("small",      covars_small),
    ("all",        covars_all),
]

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def build_batched_listdataset(batch_items, covars, freq):
    entries = []
    for tr, te in batch_items:
        entry = {"start": pd.Timestamp(tr["ds"].iloc[0]),
                 "target": tr["y"].to_numpy(dtype=np.float32)}
        if covars:
            X_past = np.stack([tr[c].to_numpy(dtype=np.float32) for c in covars], axis=0)
            X_fut  = np.stack([te[c].to_numpy(dtype=np.float32)  for c in covars], axis=0)
            entry["feat_dynamic_real"] = np.concatenate([X_past, X_fut], axis=1)
        entries.append(entry)
    return ListDataset(entries, freq=freq)

def eval_signature_bucket(bucket, covars, predictor, horizon, freq="W"):
    # bucket: list of (name, train_df, test_df) where all share the same covars list
    results = []
    for i in range(0, len(bucket), BATCH_SIZE):
        chunk = bucket[i:i+BATCH_SIZE]
        ds = build_batched_listdataset([(tr, te) for _, tr, te in chunk], covars, freq)
        preds = []
        with torch.inference_mode():
            if USE_FP16 and device == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    for fc in predictor.predict(ds):
                        preds.append(np.asarray(fc.mean, dtype=np.float32))
            else:
                for fc in predictor.predict(ds):
                    preds.append(np.asarray(fc.mean, dtype=np.float32))
        for (name, tr, te), yhat in zip(chunk, preds):
            y = te["y"].to_numpy(dtype=np.float32)
            if len(y) != horizon: y = y[-horizon:]
            results.append((name, smape_canonical(y, yhat), rmse_vec(y, yhat)))
    return results

rows, buf = [], []

def flush():
    if not buf: return
    pd.DataFrame(buf).to_csv(OUT_CSV, index=False, mode="a", header=not OUT_CSV.exists())
    buf.clear()

print(f"Running batched eval on {len(pdfs)} series with signature bucketing …")

for exog_name, covar_fn in EXOG_SETTINGS:
    # 1) Build signature buckets per horizon
    for H in HORIZONS:
        signature_buckets = defaultdict(list)  # key: tuple(covars), value: list[(name,tr,te)]
        for name, pdf in pdfs:
            covars = covar_fn(pdf)
            tr, te = split_train_test(pdf, H)
            if tr is None:  # too short
                signature_buckets[("__SKIP__",)].append((name, None, None))
                continue
            sig = tuple(covars)  # exact ordered signature
            signature_buckets[sig].append((name, tr, te))

        # 2) Evaluate each signature with its own predictor
        print(f"[{exog_name}] H={H}: {len(signature_buckets)} signature group(s)")
        for sig, bucket in signature_buckets.items():
            if sig == ("__SKIP__",):  # record skips
                for name, _, _ in bucket:
                    buf.append({"keyword": name, "exog_setting": exog_name, "horizon_weeks": H,
                                "smape_canonical": np.nan, "rmse": np.nan, "n_regressors": 0})
                continue

            covars = list(sig)
            k = len(covars)

            forecaster = MoiraiForecast(
                prediction_length=H,
                target_dim=1,
                feat_dynamic_real_dim=k,
                past_feat_dynamic_real_dim=0,
                context_length=ctx_by_H[H],
                module=module,
                patch_size="auto",
                num_samples=NUM_SAMPLES,
            )
            predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)

            results = eval_signature_bucket(bucket, covars, predictor, H, FREQ)
            for name, s, r in results:
                buf.append({
                    "keyword": name,
                    "exog_setting": exog_name,
                    "horizon_weeks": H,
                    "smape_canonical": s,
                    "rmse": r,
                    "n_regressors": k,
                })
                if len(buf) >= CHUNK_FLUSH: flush()

            del predictor, forecaster
            gc.collect()

flush()
print("✅ Done. Metrics:", OUT_CSV)



In [ ]:
# ==== Moirai Forecasts with PAST-ONLY features (lags/rolls) | Batched + FP16 ====
# - Uses ONLY engineered past features: y lags (1..12), rolling means (4,12,26), rolling std (12)
# - No future-known exog is used (feat_dynamic_real_dim=0)
# - Past-only features go into "past_feat_dynamic_real"
# - One shared feature signature for all series => blazing fast batching

from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch, gc
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# -------- Tunables --------
HORIZONS    = [1, 6, 12]
FREQ        = "W"
MAX_FILES   = None     # None = all
USE_FP16    = True     # T4 loves this
BATCH_SIZE  = 384      # try 384; if OOM, drop to 256/192
NUM_SAMPLES = 1        # mean-only; fastest for metrics
CHUNK_FLUSH = 5000     # CSV flush stride

# Engineered features (fixed signature across all series)
LAGS        = list(range(1, 13))   # y_{t-1..t-12}
ROLL_MEANS  = [4, 12, 26]          # rolling mean windows
ROLL_STDS   = [12]                  # rolling std windows
MAX_LAG     = max(LAGS + ROLL_MEANS + ROLL_STDS)

# -------- Device --------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# -------- Paths (your Colab layout) --------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_pastonly_metrics.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# -------- Helpers --------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str):
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_base(p: Path) -> pd.DataFrame | None:
    """Return pdf with ['ds','y'] sorted ascending, y=float32; None if missing columns/too short."""
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        return None
    base = (
        df_pl
        .select(pl.col("week"),
                pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
        .filter(pl.col("y").is_not_null())
    )
    pdf = base.to_pandas()
    pdf["y"] = pdf["y"].astype(np.float32, copy=False)
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    if len(pdf) < (MAX_LAG + 8):  # need enough history to build features + hold out
        return None
    return pdf

def add_past_features(pdf: pd.DataFrame) -> pd.DataFrame:
    """Create fixed past-only feature signature from y: lags, rolling means, rolling stds."""
    df = pdf.copy()
    y = df["y"]
    # Lags
    for k in LAGS:
        df[f"y_lag_{k}"] = y.shift(k)
    # Rolling means (use past-only: shift by 1 to avoid peeking)
    for w in ROLL_MEANS:
        df[f"y_rollmean_{w}"] = y.shift(1).rolling(window=w, min_periods=w).mean()
    # Rolling stds
    for w in ROLL_STDS:
        df[f"y_rollstd_{w}"] = y.shift(1).rolling(window=w, min_periods=w).std(ddof=0)
    # Drop rows that contain NaNs due to lag/rolling creation
    df = df.iloc[MAX_LAG:].reset_index(drop=True)
    # Cast features to float32
    feat_cols = [c for c in df.columns if c not in ("ds","y")]
    df[feat_cols] = df[feat_cols].astype(np.float32)
    return df

def split_train_test(df_feat: pd.DataFrame, h: int):
    if len(df_feat) <= h + 5:
        return None, None
    return df_feat.iloc[:-h].copy(), df_feat.iloc[-h:].copy()

def smape_canonical(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse_vec(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def build_batched_listdataset(batch_items, feat_order, freq):
    """batch_items: list of (train_df, test_df) — we only use train_df for past feats."""
    entries = []
    for tr, te in batch_items:
        entry = {
            "start": pd.Timestamp(tr["ds"].iloc[0]),
            "target": tr["y"].to_numpy(dtype=np.float32),  # shape [T_train]
        }
        if feat_order:
            X_past = np.stack([tr[c].to_numpy(np.float32) for c in feat_order], axis=0)  # [K, T_train]
            entry["past_feat_dynamic_real"] = X_past
        entries.append(entry)
    return ListDataset(entries, freq=freq)

# Fixed feature signature & dimensions
FEAT_ORDER = [f"y_lag_{k}" for k in LAGS] + \
             [f"y_rollmean_{w}" for w in ROLL_MEANS] + \
             [f"y_rollstd_{w}" for w in ROLL_STDS]
K = len(FEAT_ORDER)
print(f"Past-only feature dim K = {K} | features = {FEAT_ORDER}")

# -------- Preload & feature-engineer all series --------
files_all = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
files = files_all if MAX_FILES is None else files_all[:MAX_FILES]
if not files:
    raise FileNotFoundError(f"No parquet/csv files found in {IN_DIR}")

pdfs_feat = []
for p in tqdm(files, desc="Preloading + feats"):
    base = prep_keyword_base(p)
    if base is None:
        continue
    df_feat = add_past_features(base)  # adds lags/rolls and trims head
    if len(df_feat) < 24:
        continue
    pdfs_feat.append((p.stem, df_feat))
print(f"Loaded {len(pdfs_feat)} feature-ready series out of {len(files)}.")

# -------- Model (FP16 ready) --------
MODEL_ID = "Salesforce/moirai-1.1-R-small"
module = MoiraiModule.from_pretrained(MODEL_ID).to(device)
if USE_FP16 and device == "cuda":
    module = module.to(torch.float16)
module.eval()

# -------- Compute robust context length per horizon --------
ctx_by_H = {}
for H in HORIZONS:
    train_lens = [len(df_feat) - H for _, df_feat in pdfs_feat if len(df_feat) > H + 5]
    if not train_lens:
        ctx_by_H[H] = 32
        continue
    med = int(np.median(train_lens))
    p90 = int(np.percentile(train_lens, 90))
    ctx = med // 2
    ctx = int(np.clip(ctx, 32, min(512, p90)))
    ctx_by_H[H] = ctx
print("Context per horizon:", ctx_by_H)

# -------- Batched evaluation (one predictor per horizon; shared K) --------
rows, buf = [], []
def flush():
    if not buf: return
    pd.DataFrame(buf).to_csv(OUT_CSV, index=False, mode="a", header=not OUT_CSV.exists())
    buf.clear()

print(f"Running batched eval on {len(pdfs_feat)} series with past-only features …")
for H in HORIZONS:
    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=0,      # no future-known exog
        past_feat_dynamic_real_dim=K, # past-only features
        context_length=ctx_by_H[H],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)

    # Prepare splits
    items = []
    for name, df_feat in pdfs_feat:
        tr, te = split_train_test(df_feat, H)
        if tr is None:  # skip too-short series
            buf.append({"keyword": name, "exog_setting": "past_only",
                        "horizon_weeks": H, "smape_canonical": np.nan,
                        "rmse": np.nan, "n_regressors": K})
            continue
        items.append((name, tr, te))

    # Batched predict
    for i in range(0, len(items), BATCH_SIZE):
        chunk = items[i:i+BATCH_SIZE]
        ds = build_batched_listdataset([(tr, te) for _, tr, te in chunk], FEAT_ORDER, FREQ)

        preds = []
        with torch.inference_mode():
            if USE_FP16 and device == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    for fc in predictor.predict(ds):
                        preds.append(np.asarray(fc.mean, dtype=np.float32))
            else:
                for fc in predictor.predict(ds):
                    preds.append(np.asarray(fc.mean, dtype=np.float32))

        # Metrics
        for (name, tr, te), yhat in zip(chunk, preds):
            y = te["y"].to_numpy(dtype=np.float32)
            if len(y) != H: y = y[-H:]
            s = smape_canonical(y, yhat)
            r = rmse_vec(y, yhat)
            buf.append({"keyword": name, "exog_setting": "past_only",
                        "horizon_weeks": H, "smape_canonical": s,
                        "rmse": r, "n_regressors": K})
            if len(buf) >= CHUNK_FLUSH: flush()

    del predictor, forecaster
    gc.collect()

flush()
print("✅ Done. Metrics:", OUT_CSV)


In [ ]:
# ==== Evaluation for moirai_pastonly_metrics.csv ====
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ---- Paths (adjust if you changed them) ----
OUT_DIR = moirai_results_dir("dtw20")
IN_CSV  = OUT_DIR / "moirai_pastonly_metrics.csv"

# ---- Load & basic sanity ----
df = pd.read_csv(IN_CSV)
expected_cols = {"keyword","exog_setting","horizon_weeks","smape_canonical","rmse","n_regressors"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing expected columns in CSV: {missing}")

# Deduplicate (if any retries appended)
df = df.drop_duplicates(subset=["keyword","exog_setting","horizon_weeks"], keep="last").reset_index(drop=True)

# Keep only numeric rows for metrics
num = df.copy()
num["smape_canonical"] = pd.to_numeric(num["smape_canonical"], errors="coerce")
num["rmse"]            = pd.to_numeric(num["rmse"], errors="coerce")
num["horizon_weeks"]   = pd.to_numeric(num["horizon_weeks"], errors="coerce")

# Basic counts
n_total   = len(num)
n_nan_s   = num["smape_canonical"].isna().sum()
n_nan_r   = num["rmse"].isna().sum()
print(f"Rows: {n_total} | NaN sMAPE: {n_nan_s} | NaN RMSE: {n_nan_r}")

# Filter valid rows for analysis
valid = num.dropna(subset=["smape_canonical","rmse"]).copy()

# ---- Summary per horizon ----
def agg_stats(x):
    return pd.Series({
        "count": x.shape[0],
        "sMAPE_mean": x["smape_canonical"].mean(),
        "sMAPE_median": x["smape_canonical"].median(),
        "sMAPE_p10": x["smape_canonical"].quantile(0.10),
        "sMAPE_p90": x["smape_canonical"].quantile(0.90),
        "RMSE_mean": x["rmse"].mean(),
        "RMSE_median": x["rmse"].median(),
        "RMSE_p10": x["rmse"].quantile(0.10),
        "RMSE_p90": x["rmse"].quantile(0.90),
    })

summary_by_h = valid.groupby(["exog_setting","horizon_weeks"], as_index=False).apply(agg_stats)
summary_by_h = summary_by_h.sort_values(["exog_setting","horizon_weeks"])
display(summary_by_h)

# Save summary
summary_by_h_path = OUT_DIR / "eval_summary_by_horizon.csv"
summary_by_h.to_csv(summary_by_h_path, index=False)
print("Saved summary_by_horizon →", summary_by_h_path)

# ---- Leaderboards (best = lowest sMAPE) ----
TOP_N = 50  # change as you like
leaderboards = []
for h, grp in valid.groupby("horizon_weeks"):
    top = grp.sort_values("smape_canonical").head(TOP_N)[["keyword","exog_setting","horizon_weeks","smape_canonical","rmse","n_regressors"]]
    leaderboards.append(top.assign(_rank=np.arange(1, len(top)+1)))
    # Save each horizon's top table
    p = OUT_DIR / f"leaderboard_top{TOP_N}_H{int(h)}.csv"
    top.to_csv(p, index=False)
    print(f"Saved leaderboard (H={int(h)}) →", p)

lb_all = pd.concat(leaderboards, ignore_index=True)
display(lb_all.head(10))

# ---- Per-keyword best horizon (min sMAPE) ----
idx = valid.groupby("keyword")["smape_canonical"].idxmin()
best_per_kw = valid.loc[idx, ["keyword","exog_setting","horizon_weeks","smape_canonical","rmse"]].sort_values("smape_canonical").reset_index(drop=True)
display(best_per_kw.head(20))

best_per_kw_path = OUT_DIR / "best_per_keyword.csv"
best_per_kw.to_csv(best_per_kw_path, index=False)
print("Saved best_per_keyword →", best_per_kw_path)

# Distribution of chosen horizons (which H wins most often?)
h_counts = best_per_kw["horizon_weeks"].value_counts().sort_index()
print("\nChosen horizon distribution:")
print(h_counts)

# ---- Threshold report (what share beats sMAPE cutoffs?) ----
THRESHOLDS = [10, 15, 20, 25, 30, 40, 50]
rows = []
for h, grp in valid.groupby("horizon_weeks"):
    total = len(grp)
    for t in THRESHOLDS:
        share = (grp["smape_canonical"] <= t).mean() if total else np.nan
        rows.append({"horizon_weeks": int(h), "sMAPE≤": t, "share": round(share, 4), "n": total})
thr_tbl = pd.DataFrame(rows).pivot(index="sMAPE≤", columns="horizon_weeks", values="share").sort_index()
display(thr_tbl)

thr_path = OUT_DIR / "share_below_thresholds.csv"
pd.DataFrame(rows).to_csv(thr_path, index=False)
print("Saved threshold shares →", thr_path)

# ---- Quick plots (1 fig per plot, no styles/colors) ----
for h, grp in valid.groupby("horizon_weeks"):
    plt.figure(figsize=(6,4))
    plt.hist(grp["smape_canonical"].values, bins=50)
    plt.title(f"sMAPE distribution (H={int(h)})")
    plt.xlabel("sMAPE")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

for h, grp in valid.groupby("horizon_weeks"):
    plt.figure(figsize=(6,4))
    plt.hist(grp["rmse"].values, bins=50)
    plt.title(f"RMSE distribution (H={int(h)})")
    plt.xlabel("RMSE")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# ---- Optional: export a compact pivot for eyeballing in Sheets ----
pivot_compact = valid.pivot_table(index="keyword", columns="horizon_weeks", values="smape_canonical", aggfunc="min")
pivot_path = OUT_DIR / "pivot_keyword_vs_horizon_min_sMAPE.csv"
pivot_compact.to_csv(pivot_path)
print("Saved compact pivot →", pivot_path)

print("\n✅ Evaluation complete.")


In [ ]:
# ==== Moirai Multivariate (target_dim=20) | No exog | Batched + FP16 ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch, gc
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast
import matplotlib.pyplot as plt

# -------- Tunables --------
HORIZONS    = [1, 6, 12]
FREQ        = "W"
BATCH_SIZE  = 256           # try 256–384 on T4
NUM_SAMPLES = 1
USE_FP16    = True
CHUNK_FLUSH = 2000
TARGET_DIM  = 20            # enforce 20-D bundles: main + 19 neighbors
CTX_DEFAULT = 96            # longer context tends to help weekly CPC

# -------- Device --------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# -------- Paths --------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_multivar20_metrics.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# -------- Helpers --------
EXCLUDE_COLS = {"week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str):
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def select_cpc_columns(df_pl: pl.DataFrame):
    # main target
    main = ["cpc_week"] if "cpc_week" in df_pl.columns else []
    # neighbors: columns that look like CPC sims but NOT lagged features
    candidates = [c for c in df_pl.columns
                  if c.startswith("cpc_sim_") and not c.startswith("cpc_lastweek__")]
    # fallbacks: sometimes neighbors might be named 'cpc_*' but exclude lag features
    if len(candidates) < (TARGET_DIM - 1):
        extra = [c for c in df_pl.columns
                 if c.startswith("cpc_")
                 and c not in main
                 and not c.startswith("cpc_lastweek__")]
        # keep stability: prefer 'cpc_sim_' first
        seen = set(candidates)
        for c in extra:
            if len(candidates) >= (TARGET_DIM - 1): break
            if c not in seen:
                candidates.append(c); seen.add(c)
    if not main or len(candidates) < (TARGET_DIM - 1):
        return None
    # order: main, then sorted neighbors (stable)
    neigh = sorted(candidates)[:TARGET_DIM-1]
    cols = main + neigh
    return cols

def prep_multivar_matrix(p: Path):
    df_pl = _read_pl_any(p)
    cols = select_cpc_columns(df_pl)
    if cols is None: return None
    # build ds + CPC matrix
    base = (
        df_pl
        .select(["week"] + cols)
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
    )
    pdf = base.to_pandas()
    # enforce float32 and drop all-null rows across selected CPC cols
    for c in cols:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce").astype(np.float32)
    pdf = pdf.dropna(how="all", subset=cols).reset_index(drop=True)
    # require enough history
    if len(pdf) < 40: return None
    return pdf[["ds"] + cols], cols

def split_train_test_len(T, h):
    if T <= h + 5: return None
    return T - h

def smape_canonical(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse_vec(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def build_batched_listdataset(entries, freq):
    # entries: list of (Y_train [M,T_train], start_timestamp)
    gl_entries = [{"start": start_ts, "target": Yt} for (Yt, start_ts) in entries]
    return ListDataset(gl_entries, freq=freq)

# -------- Preload all series as 20-D targets --------
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
items_by_H = {H: [] for H in HORIZONS}
names_by_H = {H: [] for H in HORIZONS}

kept = 0
for p in tqdm(files, desc="Preloading multivariate 20-D"):
    res = prep_multivar_matrix(p)
    if res is None: continue
    pdf, cols = res
    # ensure exactly TARGET_DIM columns
    if len(cols) != TARGET_DIM: continue

    # construct matrix [M,T] with stable row order (cols as built)
    Y = np.stack([pdf[c].to_numpy(np.float32) for c in cols], axis=0)
    ds = pd.to_datetime(pdf["ds"].iloc[0])

    T = Y.shape[1]
    for H in HORIZONS:
        cut = split_train_test_len(T, H)
        if cut is None: continue
        Y_train = Y[:, :cut]         # [M, T_train]
        names_by_H[H].append((p.stem, cols[0]))  # track file name + main series col
        items_by_H[H].append((Y_train, pd.Timestamp(ds)))
    kept += 1

print(f"Prepared {kept} multivariate (M={TARGET_DIM}) series (per horizon)")

# -------- Model --------
MODEL_ID = "Salesforce/moirai-1.1-R-small"
module = MoiraiModule.from_pretrained(MODEL_ID).to(device)
if USE_FP16 and device == "cuda":
    module = module.to(torch.float16)
module.eval()

# -------- Context length (fixed or robust) --------
# We use a fixed longer context that works well for weekly data
ctx_by_H = {H: CTX_DEFAULT for H in HORIZONS}
print("Context per horizon:", ctx_by_H)

# -------- Batched eval (one predictor per horizon; no exog) --------
rows, buf = [], []
def flush():
    if not buf: return
    pd.DataFrame(buf).to_csv(OUT_CSV, index=False, mode="a", header=not OUT_CSV.exists())
    buf.clear()

for H in HORIZONS:
    if not items_by_H[H]:
        continue

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=TARGET_DIM,        # multivariate
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
        context_length=ctx_by_H[H],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)

    # Build test slices (for scoring) lazily to save RAM
    # We need the held-out y for the MAIN series (row 0 == cpc_week)
    test_slices = []
    for (p, _) in tqdm(names_by_H[H], desc=f"Scoring prep H={H}", leave=False):
        # reload per-file once (cheap) to get test tail; reuse the same column order
        pdf_cols = prep_multivar_matrix(Path(IN_DIR / p))  # p is stem; reconstruct full path
        # Rebuild path robustly:
        file = None
        for ext in (".parquet",".parq",".pq",".csv"):
            cand = IN_DIR / f"{p}{ext}"
            if cand.exists():
                file = cand; break
        pdf_cols = prep_multivar_matrix(file) if file else None
        if pdf_cols is None:
            test_slices.append(None)
            continue
        pdf, cols = pdf_cols
        Y_full = np.stack([pdf[c].to_numpy(np.float32) for c in cols], axis=0)
        y_main = Y_full[0]  # main target
        test_slices.append(y_main[-H:].astype(np.float32))

    # Predict in batches
    preds_main = []
    for i in range(0, len(items_by_H[H]), BATCH_SIZE):
        chunk = items_by_H[H][i:i+BATCH_SIZE]
        ds = build_batched_listdataset(chunk, FREQ)
        with torch.inference_mode():
            if USE_FP16 and device == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    for fc in predictor.predict(ds):
                        # fc.mean shape: [H, M] or [M, H] depending on lib;
                        # in uni2ts, it's [H, M]; take main (row 0 in our construction) -> column 0
                        mu = np.asarray(fc.mean, dtype=np.float32)  # [H, M]
                        preds_main.append(mu[:, 0])                  # main series
            else:
                for fc in predictor.predict(ds):
                    mu = np.asarray(fc.mean, dtype=np.float32)
                    preds_main.append(mu[:, 0])

    # Score & write
    for (name, _), yhat, ytrue in zip(names_by_H[H], preds_main, test_slices):
        if ytrue is None or len(ytrue) != H:
            s = np.nan; r = np.nan
        else:
            s = smape_canonical(ytrue, yhat)
            r = rmse_vec(ytrue, yhat)
        buf.append({
            "keyword": name,
            "exog_setting": f"multivar{TARGET_DIM}_noexog",
            "horizon_weeks": H,
            "smape_canonical": s,
            "rmse": r,
            "n_regressors": 0,
        })
        if len(buf) >= CHUNK_FLUSH: flush()

    del predictor, forecaster
    gc.collect()

flush()
print("✅ Done. Metrics:", OUT_CSV)

# Quick glance: summary
df = pd.read_csv(OUT_CSV)
df["smape_canonical"] = pd.to_numeric(df["smape_canonical"], errors="coerce")
df["rmse"] = pd.to_numeric(df["rmse"], errors="coerce")
print(df.groupby("horizon_weeks")[["smape_canonical","rmse"]].agg(["mean","median","quantile"]))


In [ ]:
# ==== Moirai | Rebuilt original + improvements (calendar + past-only lags + optional neighbor lags) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch, gc
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ---------------- Params ----------------
BEST_N     = 20
HORIZONS   = [1, 6, 12]
FREQ       = "W"
NUM_SAMPLES= 50          # (was 100 on CPU) good speed/acc trade-off
BATCH_SIZE = 64
ADD_NEIGHBOR_PAST = True # add lag-1 of top-3 similar CPCs as past-only features
MAX_NEIGH_PAST    = 3

# ---------------- Device ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ---------------- Paths ----------------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / f"moirai_exog_plus_gpu_dtw{BEST_N}.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ---------------- Feature Config ----------------
SMALL_COVARS = [
    "impressions_sum",
    "n_dev_desktop","n_dev_mobile","n_dev_tablet",
    "n_st_branded_search","n_st_generic_search",
]
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

# known-in-advance calendar (safe for future)
def add_calendar_features(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    # weekly period ~ 52.1775
    pdf["cal_woy_sin"] = np.sin(2*np.pi*woy/52.1775).astype(np.float32)
    pdf["cal_woy_cos"] = np.cos(2*np.pi*woy/52.1775).astype(np.float32)
    # simple trend: normalized week index
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/ (idx.std()+1e-8)
    return pdf

# past-only autoregressive features (no leakage)
LAGS = list(range(1,13))
ROLL_MEANS = [4,12,26]
ROLL_STDS  = [12]
def add_past_features(pdf: pd.DataFrame, target_col="y") -> pd.DataFrame:
    pdf = pdf.copy()
    for L in LAGS:
        pdf[f"y_lag_{L}"] = pdf[target_col].shift(L)
    for W in ROLL_MEANS:
        pdf[f"y_rollmean_{W}"] = pdf[target_col].rolling(W, min_periods=1).mean().shift(1)
    for W in ROLL_STDS:
        pdf[f"y_rollstd_{W}"] = pdf[target_col].rolling(W, min_periods=2).std().shift(1)
    return pdf

def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def smape_canonical(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100.0 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse_vec(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns:
        return None

    base = (
        df_pl
        .select(pl.col("week"),
                pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
        .filter(pl.col("y").is_not_null())
    )
    pdf = base.to_pandas()
    pdf["ds"] = pd.to_datetime(pdf["ds"])
    # join all other columns for potential exog
    exog_cols = [c for c in df_pl.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]
    if exog_cols:
        exog = (
            df_pl
            .select(["week"] + exog_cols)
            .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
            .drop("week")
            .sort("ds")
        ).to_pandas()
        exog["ds"] = pd.to_datetime(exog["ds"])
        pdf = pdf.merge(exog, on="ds", how="left")
        for c in exog_cols:
            pdf[c] = pd.to_numeric(pdf[c], errors="coerce").astype(np.float32)
        pdf[exog_cols] = pdf[exog_cols].ffill().bfill()

    pdf["y"] = pdf["y"].astype(np.float32, copy=False)
    pdf = pdf.sort_values("ds").reset_index(drop=True)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

# neighbor-lag: choose top-K similar CPC columns by correlation on train window
def neighbor_past_features(train_df: pd.DataFrame, full_df: pd.DataFrame) -> list[str]:
    if not ADD_NEIGHBOR_PAST:
        return []
    # candidate neighbor columns
    candidates = [c for c in full_df.columns if c.startswith("cpc_sim_") and not c.startswith("cpc_lastweek__")]
    if not candidates: return []
    y_tr = train_df["y"]
    cors = []
    for c in candidates:
        s = pd.to_numeric(train_df.get(c), errors="coerce")
        if s is None: continue
        if s.isna().all(): continue
        cor = y_tr.corr(s)
        if pd.isna(cor): continue
        cors.append((abs(cor), c))
    if not cors: return []
    top = [c for _, c in sorted(cors, reverse=True)[:MAX_NEIGH_PAST]]
    # Create lag-1 columns (will be materialized in add_past_only block)
    return top

def build_entry_univariate(tr_df, te_df, covars_future: list[str], covars_past_only: list[str]) -> dict:
    # target
    entry = {
        "start": pd.Timestamp(tr_df["ds"].iloc[0]),
        "target": tr_df["y"].to_numpy(dtype=np.float32)
    }
    # feat_dynamic_real (past+future, known-in-advance only)
    if covars_future:
        X_past = np.stack([tr_df[c].to_numpy(np.float32) for c in covars_future], axis=0)
        X_fut  = np.stack([te_df[c].to_numpy(np.float32)  for c in covars_future], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([X_past, X_fut], axis=1)
    # past_feat_dynamic_real (lags/rolls + neighbor lags; past-only)
    if covars_past_only:
        X_pastonly = np.stack([tr_df[c].to_numpy(np.float32) for c in covars_past_only], axis=0)
        entry["past_feat_dynamic_real"] = X_pastonly
    return entry

# ---- Load model once
try:
    module
except NameError:
    MODEL_ID = "Salesforce/moirai-1.1-R-small"
    module = MoiraiModule.from_pretrained(MODEL_ID).to(device).eval()

# ---- Exog settings: original + PLUS
def covars_univariate(pdf: pd.DataFrame):  return []
def covars_small(pdf: pd.DataFrame):       return [c for c in SMALL_COVARS if c in pdf.columns]
def covars_all(pdf: pd.DataFrame):         return [c for c in pdf.columns if c not in EXCLUDE_COLS and c not in ("ds","y")]

EXOG_SETTINGS = [
    ("univariate", covars_univariate),
    ("small",      covars_small),
    ("all",        covars_all),
]

def run_one(pdf: pd.DataFrame, exog_name: str, covar_fn, horizon: int):
    # base prep
    pdf_base = pdf.copy()
    # add calendar everywhere (safe future features)
    pdf_base = add_calendar_features(pdf_base)
    # choose original future exog
    orig_exog = covar_fn(pdf_base)

    # split
    tr, te = split_train_test(pdf_base, horizon)
    if tr is None: return np.nan, np.nan, 0, 0

    # build past-only lag features on y
    tr_past = add_past_features(tr, "y")
    te_past = add_past_features(pd.concat([tr.tail(max(LAGS + ROLL_MEANS + ROLL_STDS)), te], ignore_index=True), "y").iloc[len(tr):]
    past_only_cols = [c for c in tr_past.columns if c.startswith("y_lag_") or c.startswith("y_roll")]

    # neighbor lag-1s (optional)
    neigh_cols = neighbor_past_features(tr, pdf_base)
    for c in neigh_cols:
        tr_past[f"{c}_lag1"] = tr[c].shift(1)
        # for the test span we can only use past values; make them from the concat window like above
        te_concat = pd.concat([tr[[c]].tail(1), te[[c]]], ignore_index=True)  # keep alignment simple
        te_past[f"{c}_lag1"] = te_concat[c].shift(1).iloc[1:].values

    neigh_past_cols = [f"{c}_lag1" for c in neigh_cols]
    past_only_cols += neigh_past_cols

    # ensure float32 and no NaNs in past-only (drop warmup rows)
    tr_past[past_only_cols] = tr_past[past_only_cols].astype(np.float32)
    te_past[past_only_cols] = te_past[past_only_cols].astype(np.float32)
    # Drop initial rows with NA from shifting in train
    drop_n = max(LAGS + ROLL_MEANS + ROLL_STDS + [1 if neigh_cols else 0])
    if len(tr_past) <= drop_n + 5: return np.nan, np.nan, 0, 0
    tr_use = tr_past.iloc[drop_n:].copy()
    te_use = te.copy()  # for target evaluation
    # align future features to same window as tr_use
    tr_base_aligned = pdf_base.iloc[:len(tr_past)].iloc[drop_n:].copy()
    te_base = pdf_base.iloc[len(tr):].copy()

    # Future-known covars: original exog + calendar
    future_covars = [c for c in orig_exog if c in tr_base_aligned.columns]
    future_covars += ["cal_woy_sin","cal_woy_cos","cal_trend"]

    # Build ListDataset entry
    entry = build_entry_univariate(tr_base_aligned.assign(**{k: tr_use[k] for k in past_only_cols}),
                                   te_base,
                                   covars_future=future_covars,
                                   covars_past_only=past_only_cols)

    ctx = int(np.clip(len(tr_base_aligned)//2, 32, len(tr_base_aligned)))
    forecaster = MoiraiForecast(
        prediction_length=horizon,
        target_dim=1,
        feat_dynamic_real_dim=len(future_covars),
        past_feat_dynamic_real_dim=len(past_only_cols),
        context_length=ctx,
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    ds = ListDataset([entry], freq=FREQ)
    fc = next(predictor.predict(ds))
    yhat = np.asarray(fc.mean, dtype=np.float32)
    ytrue = te_use["y"].to_numpy(np.float32)
    if len(ytrue) != horizon:
        ytrue = ytrue[-horizon:]
    s = smape_canonical(ytrue, yhat)
    r = rmse_vec(ytrue, yhat)
    return s, r, len(future_covars), len(past_only_cols)

# --------------- Run ---------------
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
if not files:
    raise FileNotFoundError(f"No parquet/csv files found in {IN_DIR}")

rows, CHUNK = [], 250
def flush(buf):
    if not buf: return
    df = pd.DataFrame(buf)
    mode = "a" if OUT_CSV.exists() else "w"
    df.to_csv(OUT_CSV, index=False, mode=mode, header=not OUT_CSV.exists())
    buf.clear()

print(f"Found {len(files)} files.")
for i, p in enumerate(tqdm(files, desc=f"DTW={BEST_N} | horizons={HORIZONS} | exog=univariate/small/all + PLUS (GPU)")):
    pdf = prep_keyword_pdf(p)
    if pdf is None or len(pdf) < 24:  # need room for lags
        continue
    for exog_name, covar_fn in EXOG_SETTINGS:
        for H in HORIZONS:
            try:
                s, r, k_future, k_past = run_one(pdf, exog_name, covar_fn, H)
            except Exception as e:
                print(f"[WARN] {p.stem} | {exog_name} | H={H} failed: {e}")
                s, r, k_future, k_past = np.nan, np.nan, 0, 0
            rows.append({
                "keyword": p.stem,
                "exog_setting": exog_name + "+plus",
                "horizon_weeks": H,
                "smape_canonical": s,
                "rmse": r,
                "n_future_covars": k_future,
                "n_past_covars": k_past,
                "neighbor_past": int(ADD_NEIGHBOR_PAST),
            })
    if (i+1) % CHUNK == 0:
        flush(rows)

flush(rows)
print("✅ Done. Metrics:", OUT_CSV)

# quick sanity peek
df = pd.read_csv(OUT_CSV)
for h in HORIZONS:
    g = df[df.horizon_weeks==h]
    print(f"H={h}: sMAPE mean={g['smape_canonical'].mean():.2f}, median={g['smape_canonical'].median():.2f} | RMSE mean={g['rmse'].mean():.4f}")


### ↓ Canonical cell for the paper (see banner at the top of the notebook)

Below is the cell whose output is reported in the paper. All earlier cells in this notebook run, pin packages, or iterate the methodology that leads up to this cell.


In [ ]:
# ==== Moirai H=6 (auto-scales for 110–127 weeks) ====
from pathlib import Path
from datetime import date
import numpy as np, pandas as pd, polars as pl
from tqdm.auto import tqdm
import torch
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai import MoiraiModule, MoiraiForecast

# ------------ Fixed eval params ------------
FREQ        = "W"
H           = 6
NUM_SAMPLES = 200        # 120–300: higher = smoother median, slower
BATCH_SIZE  = 64
CTX_CAP     = 96         # cap context for speed/stability
CTX_FLOOR   =  FortyEight = 48

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
torch.set_default_dtype(torch.float32)
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("✅ Using device:", device)

# ------------ Paths ------------
IN_DIR  = DTW_DFS_DIR / "dtw_20_dfs"
OUT_DIR = moirai_results_dir("dtw20")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "moirai_H6_auto_adaptive.csv"
print("Input dir:", IN_DIR)
print("Output   :", OUT_CSV)

# ------------ IO + helpers ------------
EXCLUDE_COLS = {"week","cpc_week","keyword","adcost_sum","adclicks_sum","year","week_num"}

def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def _read_pl_any(p: Path) -> pl.DataFrame:
    sfx = p.suffix.lower()
    if sfx in (".parquet",".parq",".pq"): return pl.read_parquet(p)
    if sfx == ".csv":                     return pl.read_csv(p)
    raise ValueError(f"Unsupported file type: {p}")

def prep_keyword_pdf(p: Path) -> pd.DataFrame | None:
    df_pl = _read_pl_any(p)
    if "week" not in df_pl.columns or "cpc_week" not in df_pl.columns: return None
    base = (
        df_pl.select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64, strict=False).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week").sort("ds").filter(pl.col("y").is_not_null())
    ).to_pandas()
    base["ds"] = pd.to_datetime(base["ds"])
    base["y"]  = pd.to_numeric(base["y"], errors="coerce").astype(np.float32)
    base = base.sort_values("ds").reset_index(drop=True)
    return base if len(base) >= (H+32) else None

def add_calendar(pdf: pd.DataFrame, k=3) -> pd.DataFrame:
    pdf = pdf.copy()
    woy = pdf["ds"].dt.isocalendar().week.astype(int)
    period = 52.1775
    for i in range(1, k+1):
        pdf[f"cal_sin_{i}"] = np.sin(2*np.pi*i*woy/period).astype(np.float32)
        pdf[f"cal_cos_{i}"] = np.cos(2*np.pi*i*woy/period).astype(np.float32)
    idx = np.arange(len(pdf), dtype=np.float32)
    pdf["cal_trend"] = (idx - idx.mean())/(idx.std()+1e-8)
    return pdf

def split_train_test(pdf: pd.DataFrame, h: int):
    if len(pdf) <= h + 5: return None, None
    return pdf.iloc[:-h].copy(), pdf.iloc[-h:].copy()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    denom = (np.abs(y_true) + np.abs(y_pred))/2.0
    m = denom != 0
    return float(100.0*np.mean(np.abs(y_pred[m]-y_true[m])/denom[m]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, np.float32)
    y_pred = np.asarray(y_pred, np.float32)
    return float(np.sqrt(np.mean((y_pred-y_true)**2)))

# ---- Model (load once)
try:
    module
except NameError:
    module = MoiraiModule.from_pretrained("Salesforce/moirai-1.1-R-small").to(device).eval()

def make_auto_feature_plan(n_total: int, h: int):
    """Return dict with adaptive windows given total length."""
    n_train = n_total - h
    # lag budget ≈ 20% of train (min 8, max 26)
    max_lag = int(max(8, min(26, 0.2*n_train)))
    # rolling windows: subset of {6,12,26,52} that fit history
    roll_candidates = [6,12,26,52]
    roll_means = [w for w in roll_candidates if w <= max(max_lag, 12)]
    roll_stds  = [w for w in [6,12,26] if w <= max(max_lag, 12)]
    # EMAs: safe short spans
    ema_spans = [s for s in [6,12] if s <= max_lag]
    # warmup: ensure enough to compute features (cap to 1/3 of train)
    warm = max(max_lag, max(roll_means+[6]), max(roll_stds+[6]), max(ema_spans+[6]))
    warm = int(min(warm, max(12, n_train//3)))
    # context: use ~70% of post-warm train, bounded
    ctx = n_train - warm
    ctx = max(CTX_FLOOR, min(CTX_CAP, int(0.7*ctx)))
    return {
        "lags": list(range(1, max_lag+1)),
        "roll_means": roll_means,
        "roll_stds": roll_stds,
        "ema_spans": ema_spans,
        "warm": warm,
        "ctx": ctx,
        "neigh_lags": list(range(1, min(6, max_lag)+1))
    }

def add_past_feats(pdf: pd.DataFrame, target: str, plan: dict) -> pd.DataFrame:
    pdf = pdf.copy()
    for L in plan["lags"]:
        pdf[f"{target}_lag_{L}"] = pdf[target].shift(L)
    for W in plan["roll_means"]:
        pdf[f"{target}_rollmean_{W}"] = pdf[target].rolling(W, min_periods=1).mean().shift(1)
    for W in plan["roll_stds"]:
        pdf[f"{target}_rollstd_{W}"] = pdf[target].rolling(W, min_periods=2).std().shift(1)
    for S in plan["ema_spans"]:
        pdf[f"{target}_ema_{S}"] = pdf[target].ewm(span=S, min_periods=1, adjust=False).mean().shift(1)
    return pdf

def pick_top_neighbors(tr: pd.DataFrame, k=3):
    # Use available cpc_sim_* columns (if any), pick by corr on TRAIN ONLY
    cand = [c for c in tr.columns if c.startswith("cpc_sim_") and "lastweek" not in c]
    if not cand: return []
    y = tr["y"]
    cors=[]
    for c in cand:
        s = pd.to_numeric(tr[c], errors="coerce")
        if s.isna().all(): continue
        r = y.corr(s)
        if pd.isna(r): continue
        cors.append((abs(r), c))
    cors.sort(reverse=True)
    return [c for _,c in cors[:k]]

def run_one(pdf: pd.DataFrame):
    # Calendar (future-known)
    pdf = add_calendar(pdf, k=3)

    # Log1p transform target for stability; keep original y for eval
    pdf = pdf.copy()
    pdf["y_orig"] = pdf["y"].astype(np.float32)
    pdf["y"] = np.log1p(np.clip(pdf["y_orig"], a_min=0, a_max=None)).astype(np.float32)

    tr, te = split_train_test(pdf, H)
    if tr is None: return np.nan, np.nan, 0, 0

    plan = make_auto_feature_plan(len(pdf), H)

    # Build past features on y (log-space)
    tr_p = add_past_feats(tr, "y", plan)
    te_p = add_past_feats(pd.concat([tr.tail(plan["warm"]), te], ignore_index=True), "y", plan).iloc[plan["warm"]:]

    # Neighbor past lags (if sim columns exist) — no leakage
    neigh = pick_top_neighbors(tr, k=3)
    for c in neigh:
        for L in plan["neigh_lags"]:
            tr_p[f"{c}_lag{L}"] = tr[c].shift(L)
            te_concat = pd.concat([tr[[c]].tail(L), te[[c]]], ignore_index=True)
            te_p[f"{c}_lag{L}"] = te_concat[c].shift(L).iloc[L:].values

    past_cols = [c for c in tr_p.columns if c.startswith(("y_lag_","y_rollmean_","y_rollstd_","y_ema_"))] + \
                [f"{c}_lag{L}" for c in neigh for L in plan["neigh_lags"]]

    # Drop warmup
    if len(tr_p) <= plan["warm"] + 8:
        return np.nan, np.nan, 0, 0
    tr_base = tr.iloc[plan["warm"]:].copy()
    tr_p    = tr_p.iloc[plan["warm"]:].copy()
    te_base = te.copy()

    # Future-known covars: calendar only (safe)
    fut_cols = [c for c in pdf.columns if c.startswith("cal_")]

    entry = {
        "start": pd.Timestamp(tr_base["ds"].iloc[0]),
        "target": tr_base["y"].to_numpy(np.float32),          # log-space
    }
    if fut_cols:
        Xp = np.stack([tr_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        Xf = np.stack([te_base[c].to_numpy(np.float32) for c in fut_cols], axis=0)
        entry["feat_dynamic_real"] = np.concatenate([Xp, Xf], axis=1)
    if past_cols:
        entry["past_feat_dynamic_real"] = np.stack([tr_p[c].to_numpy(np.float32) for c in past_cols], axis=0)

    forecaster = MoiraiForecast(
        prediction_length=H,
        target_dim=1,
        feat_dynamic_real_dim=len(fut_cols),
        past_feat_dynamic_real_dim=len(past_cols),
        context_length=plan["ctx"],
        module=module,
        patch_size="auto",
        num_samples=NUM_SAMPLES,
    )
    predictor = forecaster.create_predictor(batch_size=BATCH_SIZE, device=device)
    fc = next(predictor.predict(ListDataset([entry], freq=FREQ)))

    # Median over samples in log-space → inverse-transform to original space
    pred_log = np.asarray(np.median(fc.samples, axis=0), dtype=np.float32) if fc.samples is not None else np.asarray(fc.mean, dtype=np.float32)
    yhat = np.expm1(pred_log)                                  # back to original scale
    y    = te_base["y_orig"].to_numpy(np.float32)              # original target for eval
    return smape(y, yhat), rmse(y, yhat), len(fut_cols), len(past_cols)

# ------------ Run (H=6 only) ------------
rows=[]
files = sorted([p for p in IN_DIR.glob("*") if p.suffix.lower() in (".parquet",".parq",".pq",".csv")])
print(f"Found {len(files)} files.")
for p in tqdm(files, desc="H=6 auto-adaptive"):
    pdf = prep_keyword_pdf(p)
    if pdf is None:
        continue
    try:
        s, r, kf, kp = run_one(pdf)
    except Exception as e:
        print(f"[WARN] {p.stem} failed: {e}")
        s, r, kf, kp = np.nan, np.nan, 0, 0
    rows.append({
        "keyword": p.stem, "exog_setting": "auto+past_only+calendar",
        "horizon_weeks": H, "smape_canonical": s, "rmse": r,
        "n_future_covars": kf, "n_past_covars": kp, "n_total": len(pdf)
    })

out = pd.DataFrame(rows)
out.to_csv(OUT_CSV, index=False)
print("✅ Done. Metrics:", OUT_CSV)

valid = out.dropna(subset=["smape_canonical","rmse"])
if not valid.empty:
    g = valid.agg(
        count=("rmse","count"),
        sMAPE_mean=("smape_canonical","mean"),
        sMAPE_median=("smape_canonical","median"),
        RMSE_mean=("rmse","mean"),
        RMSE_median=("rmse","median"),
    )
    display(g)
    # quick percent-below thresholds
    for t in [10,15,20,25,30,40]:
        share = (valid["smape_canonical"] <= t).mean()
        print(f"sMAPE≤{t:>2}: {share:.3f}")
else:
    print("No valid rows.")
